# topk-predictions — faded example 1: Top-5 hit mask via any

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `topk-predictions`. Running the beacon reports progress on the `Eval: topk predictions` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: topk predictions` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`topk-predictions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "topk-predictions"
DD_SUBTOPIC = "Eval: topk predictions"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A top-k hit mask is built by comparing the `(B, k)` top-k indices against the broadcast labels `(B, 1)` and reducing the k axis with `.any`. The result is a `(B,)` boolean: True where the true label appears among the top k predictions.

## Faded exercise 1

### Build the top-5 hit mask

Implement `top5_hits(logits, labels)` returning a `(B,)` boolean tensor: True for each sample whose true label is among its 5 highest-scoring classes. The topk indices are computed for you; complete the line that turns them and `labels` into the per-sample hit mask.

**Fill in:** comparing topk indices against broadcast labels and reducing the k axis with any

In [ ]:
import torch as t

def top5_hits(logits: t.Tensor, labels: t.Tensor) -> t.Tensor:
    idx = logits.topk(5, dim=-1).indices  # (B, 5)
    hits = (idx == labels.unsqueeze(-1)).any(dim=-1)
    return hits

t.manual_seed(0)
print(top5_hits(t.randn(4, 10), t.randint(0, 10, (4,))))


def _test():
    # hand-built logits so the top-5 set is known
    # row0: scores favor classes 9,8,7,6,5 ; label 7 is in top5 -> hit
    # row1: label 0 is the smallest score -> miss
    logits = t.tensor([
        [0., 1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [9., 8., 7., 6., 5., 4., 3., 2., 1., 0.],
    ])
    labels = t.tensor([7, 9])  # row0: 7 in top5 (hit); row1: 9 is smallest (miss)
    hits = top5_hits(logits, labels)
    assert hits.dtype == t.bool, hits.dtype
    assert hits.shape == (2,), hits.shape
    # independent truth: top5 of row0 is {5,6,7,8,9}; row1 is {0,1,2,3,4}
    assert bool(hits[0]) is True
    assert bool(hits[1]) is False


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def top5_hits(logits: t.Tensor, labels: t.Tensor) -> t.Tensor:
    idx = logits.topk(5, dim=-1).indices  # (B, 5)
    hits = (idx == labels.unsqueeze(-1)).any(dim=-1)
    return hits

t.manual_seed(0)
print(top5_hits(t.randn(4, 10), t.randint(0, 10, (4,))))
```
</details>